<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/%D0%9D%D0%BE%D1%83%D1%82%D0%B1%D1%83%D0%BA_2_%D0%BE%D1%81%D0%BD%D0%BE%D0%B2%D0%BD%D0%BE%D0%B9_%D1%8D%D0%BA%D1%81%D0%BF%D0%B5%D1%80%D0%B8%D0%BC%D0%B5%D0%BD%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ноутбук 2 — Основной MARL-эксперимент по 9 регионам СФО

Многоагентное управление демографией 9 регионов Сибирского федерального округа
на реалистичном CCM-бейзлайне (выход Ноутбука 1).

## Состав эксперимента
1. **Среда** из CCM-панелей + **world-model** (ансамбль 5 вероятностных MLP, NLL-обучение).
2. **8 агентов**, непрерывные действия [−1, 1]:
   - институциональные: `RegionalGov`, `HealthcareSystem`, `EducationInfra`, `MigrationPolicy`;
   - поведенческие: `Women`, `Households`, `Families`, `Migrants`.
3. **MADDPG и MATD3** — оба с **ранней остановкой** и загрузкой **лучшего по награде чекпойнта**.
4. **Самореферентный MATD3**: эволюция reward-весов + интенсивности действий
   (популяция 8→12 адаптивно, кроссовер+мутации, отдельный прогон по каждому региону).
5. **Страховка от reward hacking** (двойная): эволюция + внешняя hold-out валидация
   с штрафом за неправдоподобность.
6. **Мульти-сид (10 сидов)** + 95% ДИ: эффект **доказан**, если ДИ не пересекает 0.
7. **Финал** — общий частично самореферентный агент, обучающийся на опыте прошлых прогонов.

## Утверждённые параметры
`N_SEEDS=10`, `POP_MIN=8 → POP_MAX=12`, `TFR_CAP=2.5`, `MAX_EPISODES=600` (ранняя остановка раньше),
runtime = **Google Colab T4**. Тяжёлые самореферентные прогоны режутся на чекпойнты (resume между сессиями).

## Зависимости и данные
Требуется `panels_all.csv` / `panel_<регион>.csv` из Ноутбука 1. На Colab T4 включите GPU
(`Среда выполнения → Сменить среду выполнения → T4 GPU`).

In [ ]:
!pip -q install torch statsmodels scikit-learn scipy 2>/dev/null
print('зависимости готовы')

зависимости готовы


In [ ]:
# ---------------------------------------------------------------------------
# ВСПОМОГАТЕЛЬНЫЙ АККУРАТНЫЙ PROGRESS BAR ДЛЯ ДЛИТЕЛЬНЫХ ИССЛЕДОВАНИЙ
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm
from contextlib import nullcontext

def pbar(iterable=None, total=None, desc="", leave=True, disable=False, position=None):
    return tqdm(
        iterable=iterable,
        total=total,
        desc=desc,
        leave=leave,
        disable=disable,
        position=position,
        dynamic_ncols=True,
        mininterval=0.5,
        smoothing=0.12,
        bar_format="{desc}: {percentage:6.2f}%|{bar}| {n_fmt}/{total_fmt} "
                   "[{elapsed}<{remaining}, {rate_fmt}{postfix}]"
    )

print("OK: progress bar helper загружен")

OK: progress bar helper загружен


## 1. Конфигурация и гиперпараметры
Утверждённые параметры эксперимента: 10 сидов, популяция 8→12, кэп СКР 2.5, потолок 600 эпизодов с ранней остановкой.

In [ ]:
# -*- coding: utf-8 -*-
"""
НОУТБУК 2 — Основной MARL-эксперимент по 9 регионам СФО
========================================================
Состав (по утверждённому плану):
  • Среда из CCM-панелей (выход Ноутбука 1), world-model = ансамбль 5 вероятностных MLP;
  • 8 агентов (4 институциональных + 4 поведенческих), непрерывные действия [-1, 1];
  • MADDPG и MATD3 — оба с РАННЕЙ ОСТАНОВКОЙ и загрузкой ЛУЧШЕГО по награде чекпойнта;
  • Самореферентный модуль для MATD3: эволюция reward-весов + интенсивности действий
    (популяция 8→12 адаптивно, кроссовер+мутации, по регионам) — Darwin-Gödel-направление;
  • Страховка от reward hacking: (1) эволюция + (2) внешняя hold-out валидация + штраф
    за неправдоподобность;
  • Мульти-сид (10 сидов), 95% ДИ изменения метрики не пересекает ноль = доказанный эффект;
  • Финал — общий частично самореферентный агент, обучающийся на опыте прошлых прогонов.

Параметры утверждены пользователем:
  POP_MIN=8, POP_MAX=12 (адаптивно), N_SEEDS=10, TFR_CAP=2.5, runtime = Colab T4.
"""
import numpy as np, math, copy, time, json, os
from dataclasses import dataclass, field
import torch, torch.nn as nn, torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- утверждённые гиперпараметры ---
N_SEEDS      = 10           # сидов на конфигурацию (статистика + 95% ДИ)
POP_MIN      = 8            # стартовый размер эволюционной популяции
POP_MAX      = 12           # потолок (адаптивное расширение 8->12)
TFR_CAP      = 2.5          # мягкий кэп СКР (как в Ноутбуке 1)
MAX_EPISODES = 600          # потолок эпизодов (ранняя остановка обычно раньше)
ES_WINDOW    = 20           # окно скользящего среднего награды (ранняя остановка)
ES_PATIENCE  = 5            # терпение в окнах: стоп, если 5*окно без улучшения
CKPT_EVERY   = 25           # частота чекпойнтинга (для best-reward загрузки и resume)
ROLLOUTS     = 30           # усреднение по роллаутам world-model
SHOCK_STD    = 0.02         # стохастический шок world-model

# --- 8 агентов ---
INSTITUTIONAL = ["RegionalGov", "HealthcareSystem", "EducationInfra", "MigrationPolicy"]
BEHAVIORAL    = ["Women", "Households", "Families", "Migrants"]
AGENTS = INSTITUTIONAL + BEHAVIORAL
N_AGENTS = len(AGENTS)

# --- стартовые веса reward (как в курской референсной работе) ---
REWARD_WEIGHTS_INIT = dict(pop=0.20, tfr=0.25, mig=0.15, nat=0.15,
                           housing=0.10, obgyn=0.10, action_cost=-0.05)

def set_seed(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print(f"OK: конфигурация загружена | device={DEVICE} | агентов={N_AGENTS} | сидов={N_SEEDS}")

OK: конфигурация загружена | device=cuda | агентов=8 | сидов=10


## 2. World-model: ансамбль 5 вероятностных MLP
Предсказывает дельту демографического состояния по (состояние, совместное действие). Обучается на NLL; разброс между членами ансамбля = эпистемическая неопределённость (используется в anti-reward-hacking).

In [ ]:
# ---------------------------------------------------------------------------
# WORLD-MODEL: ансамбль 5 вероятностных MLP (предсказание дельты состояния)
# ---------------------------------------------------------------------------
class ProbMLP(nn.Module):
    """Вероятностный MLP: выдаёт mean и log_var дельты состояния (NLL-обучение)."""
    def __init__(self, in_dim, out_dim, hidden=256):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU())
        self.mean = nn.Linear(hidden, out_dim)
        self.logv = nn.Linear(hidden, out_dim)
    def forward(self, x):
        h = self.body(x)
        return self.mean(h), torch.clamp(self.logv(h), -10, 4)

class WorldModelEnsemble:
    """
    Ансамбль из 5 ProbMLP. Предсказывает дельту демографического состояния
    s_{t+1}-s_t по (s_t, совместное действие). Эпистемическая неопределённость —
    разброс между членами ансамбля (используется в anti-reward-hacking).
    """
    def __init__(self, state_dim, action_dim, n_models=5, lr=1e-3):
        self.models = [ProbMLP(state_dim+action_dim, state_dim).to(DEVICE) for _ in range(n_models)]
        self.opts = [torch.optim.Adam(m.parameters(), lr=lr) for m in self.models]
        self.state_dim, self.action_dim = state_dim, action_dim
        self.s_mu = np.zeros(state_dim); self.s_sd = np.ones(state_dim)

    def fit(self, S, A, S_next, epochs=40, batch=128, verbose=False):
        """Обучение на исторических переходах. NLL-функция потерь, bootstrap по членам."""
        self.s_mu, self.s_sd = S.mean(0), S.std(0)+1e-6
        Sn = (S-self.s_mu)/self.s_sd; Snn = (S_next-self.s_mu)/self.s_sd
        dY = Snn - Sn
        X = np.concatenate([Sn, A], axis=1).astype(np.float32)
        Y = dY.astype(np.float32)
        X = torch.tensor(X, device=DEVICE); Y = torch.tensor(Y, device=DEVICE)
        n = len(X)
        for mi, (m, opt) in enumerate(zip(self.models, self.opts)):
            idx_boot = np.random.choice(n, n, replace=True)  # bootstrap
            for ep in range(epochs):
                perm = np.random.permutation(idx_boot)
                for i in range(0, n, batch):
                    bidx = perm[i:i+batch]
                    mu, logv = m(X[bidx])
                    inv = torch.exp(-logv)
                    loss = (0.5*((Y[bidx]-mu)**2*inv + logv)).mean()
                    opt.zero_grad(); loss.backward(); opt.step()
            if verbose: print(f"  world-model {mi}: финальный NLL={loss.item():.4f}")

    def step(self, s, a, shock=SHOCK_STD):
        """Один шаг: возвращает s_next и эпистемическую неопределённость (разброс ансамбля)."""
        sn = (s-self.s_mu)/self.s_sd
        x = torch.tensor(np.concatenate([sn, a]).astype(np.float32), device=DEVICE).unsqueeze(0)
        deltas = []
        with torch.no_grad():
            for m in self.models:
                mu, logv = m(x)
                eps = torch.randn_like(mu)*torch.exp(0.5*logv)*shock
                deltas.append((mu+eps).cpu().numpy()[0])
        deltas = np.array(deltas)
        delta = deltas.mean(0); epi = deltas.std(0).mean()      # эпистемич. неопр.
        s_next = s + delta*self.s_sd                              # денормализация дельты
        return s_next, epi

print("OK: WorldModelEnsemble загружен")

OK: WorldModelEnsemble загружен


## 3. Среда `DemographyEnv`
Эпизод = прогон 2025→2050. Reward считается **относительно нелинейного CCM-baseline** (а не линии — это и есть фикс корневой причины прошлого провала). Кэп СКР 2.5 + штраф за эпистемич. неопределённость встроены в reward.

In [ ]:
# ---------------------------------------------------------------------------
# СРЕДА: DemographyEnv поверх CCM-панели и world-model
# ---------------------------------------------------------------------------
# Индексы целевых индикаторов в state (имена -> позиции задаются при загрузке панели)
TARGET_KEYS = ["Численность населения всего", "СКР (всего)",
               "Коэффициент миграционного прироста (на 10000)",
               "Коэффициент естественного прироста (на 1000)",
               "Общая площадь жилья на 1 жителя (кв. м)",
               "Укомплектованность акушерами-гинекологами (%)"]

# отображение 8 агентов -> подмножества индикаторов состояния, на которые влияют действия
# (институциональные двигают инфраструктуру/политику, поведенческие — поведение)
AGENT_LEVERS = {
    "RegionalGov":      ["Общая площадь жилья на 1 жителя (кв. м)", "Число молодых семей, улучшивших жилищные условия"],
    "HealthcareSystem": ["Укомплектованность акушерами-гинекологами (%)", "Укомплектованность неонатологами (%)", "Число циклов ЭКО"],
    "EducationInfra":   ["Валовой коэффициент охвата дошкольным образованием (%)", "Обеспеченность местами в ДОУ (на 1000 детей)"],
    "MigrationPolicy":  ["Миграционное сальдо", "Коэффициент миграционного прироста (на 10000)"],
    "Women":            ["СКР (всего)", "Средний возраст матери при рождении ребёнка"],
    "Households":       ["Коэффициент брачности (на 1000)", "Доля рождений в браке"],
    "Families":         ["СКР третьих и последующих детей", "Число многодетных семей, улучшивших жилищные условия"],
    "Migrants":         ["Численность мигрантов трудоспособного возраста (сальдо)"],
}

class DemographyEnv:
    """
    Эпизод = прогон 2025->2050 по годам. Состояние s = вектор индикаторов (54-1=53 числовых).
    Совместное действие = вектор [-1,1] по 8 агентам, масштабируется в воздействие на
    индикаторы-рычаги через action_intensity. Reward = взвешенная сумма прогресса по целям.
    """
    def __init__(self, panel_df, world_model, col_index, reward_weights=None,
                 action_intensity=None, horizon_years=(2025, 2050)):
        self.panel = panel_df.reset_index(drop=True)
        self.cols = [c for c in panel_df.columns if c not in ("Год","Регион","Тип половозрастной структуры (Сундберг)")]
        self.col_index = col_index            # имя -> позиция в state-векторе
        self.wm = world_model
        self.w = dict(reward_weights or REWARD_WEIGHTS_INIT)
        self.intensity = float(action_intensity if action_intensity is not None else 1.0)
        self.y0, self.y1 = horizon_years
        self.state_dim = len(self.cols); self.action_dim = N_AGENTS
        self._build_baseline()

    def _row(self, year):
        r = self.panel[self.panel["Год"]==year]
        return r[self.cols].to_numpy()[0].astype(float)

    def _build_baseline(self):
        """Baseline-траектория = CCM-панель БЕЗ воздействия (политики = 0)."""
        self.baseline = {y: self._row(y) for y in range(self.y0, self.y1+1)}

    def reset(self):
        self.year = self.y0
        self.s = self._row(self.y0).copy()
        return self.s.copy()

    def _apply_actions(self, action):
        """Действия [-1,1] -> модификаторы индикаторов-рычагов (с учётом intensity)."""
        mod = np.zeros(self.state_dim)
        for ai, agent in enumerate(AGENTS):
            a = float(np.clip(action[ai], -1, 1)) * self.intensity
            for lever in AGENT_LEVERS[agent]:
                if lever in self.col_index:
                    j = self.col_index[lever]
                    mod[j] += a * 0.03 * abs(self.s[j] if self.s[j]!=0 else 1.0)  # 3% рычаг
        return mod

    def step(self, action):
        mod = self._apply_actions(action)
        s_in = self.s + mod
        s_next, epi = self.wm.step(s_in, np.clip(action,-1,1))
        # кэп СКР 2.5 (страховка от reward hacking через невозможные значения)
        if "СКР (всего)" in self.col_index:
            j = self.col_index["СКР (всего)"]; s_next[j] = np.clip(s_next[j], 0.7, TFR_CAP)
        self.year += 1
        base = self.baseline.get(self.year, s_next)
        reward, info = self._reward(s_next, base, action, epi)
        self.s = s_next
        done = self.year >= self.y1
        return s_next.copy(), reward, done, info

    def _reward(self, s, base, action, epi):
        """Reward = прогресс к целям ОТНОСИТЕЛЬНО CCM-baseline (нелинейного!)."""
        def rel(key):
            if key not in self.col_index: return 0.0
            j = self.col_index[key]; b = base[j]
            return (s[j]-b)/(abs(b)+1e-6)
        w = self.w
        r = (w["pop"]*rel("Численность населения всего")
             + w["tfr"]*rel("СКР (всего)")
             + w["mig"]*rel("Коэффициент миграционного прироста (на 10000)")
             + w["nat"]*rel("Коэффициент естественного прироста (на 1000)")
             + w["housing"]*rel("Общая площадь жилья на 1 жителя (кв. м)")
             + w["obgyn"]*rel("Укомплектованность акушерами-гинекологами (%)")
             + w["action_cost"]*float(np.mean(np.abs(action))))
        # штраф за эпистемическую неопределённость (anti-reward-hacking: не доверяем
        # участкам пространства, где world-model не уверена)
        r -= 0.5*float(epi)
        return r, dict(epi=epi)

print("OK: DemographyEnv загружен")

OK: DemographyEnv загружен


## 4. Сети агентов + MADDPG / MATD3
Централизованный критик (видит все действия), децентрализованные акторы. MATD3 = twin-критики (min из 2) + сглаживание целевого действия + отложенное обновление актора (защита от переоценки Q).

In [ ]:
# ---------------------------------------------------------------------------
# СЕТИ АГЕНТОВ + MADDPG / MATD3 (общий критик, централизованное обучение)
# ---------------------------------------------------------------------------
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim=1, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(s_dim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, a_dim), nn.Tanh())  # действие в [-1,1]
    def forward(self, s): return self.net(s)

class Critic(nn.Module):
    """Централизованный критик: вход = глобальное состояние + ВСЕ действия."""
    def __init__(self, s_dim, total_a, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(s_dim+total_a, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, 1))
    def forward(self, s, a): return self.net(torch.cat([s, a], dim=-1))

class ReplayBuffer:
    def __init__(self, cap=100000): self.cap=cap; self.buf=[]; self.pos=0
    def add(self, *tr):
        if len(self.buf)<self.cap: self.buf.append(tr)
        else: self.buf[self.pos]=tr; self.pos=(self.pos+1)%self.cap
    def sample(self, n):
        idx=np.random.choice(len(self.buf), min(n,len(self.buf)), replace=False)
        return [self.buf[i] for i in idx]
    def __len__(self): return len(self.buf)

class MARLTrainer:
    """
    Унифицированный тренер MADDPG / MATD3.
      algo='maddpg' : классический DDPG-критик.
      algo='matd3'  : TWIN-критики (min из 2) + сглаживание целевого действия +
                      отложенное обновление актора (td3-трюки против переоценки Q).
    Оба поддерживают:
      • РАННЮЮ ОСТАНОВКУ по скользящему среднему валидационной награды;
      • загрузку ЛУЧШЕГО по награде чекпойнта в финал.
    """
    def __init__(self, env_fn, algo="matd3", seed=0, gamma=0.95, tau=0.01, lr=1e-3,
                 policy_delay=2, target_noise=0.2, noise_clip=0.5):
        self.env_fn=env_fn; self.algo=algo; self.seed=seed
        self.gamma=gamma; self.tau=tau; self.policy_delay=policy_delay
        self.target_noise=target_noise; self.noise_clip=noise_clip
        set_seed(seed)
        env=env_fn(); self.s_dim=env.state_dim; self.n=N_AGENTS
        self.actors=[Actor(self.s_dim).to(DEVICE) for _ in range(self.n)]
        self.actors_t=[copy.deepcopy(a) for a in self.actors]
        total_a=self.n
        self.critic1=Critic(self.s_dim, total_a).to(DEVICE)
        self.critic1_t=copy.deepcopy(self.critic1)
        self.critic2=Critic(self.s_dim, total_a).to(DEVICE) if algo=="matd3" else None
        self.critic2_t=copy.deepcopy(self.critic2) if self.critic2 is not None else None
        self.a_opts=[torch.optim.Adam(a.parameters(), lr=lr) for a in self.actors]
        cps=list(self.critic1.parameters())+(list(self.critic2.parameters()) if self.critic2 is not None else [])
        self.c_opt=torch.optim.Adam(cps, lr=lr)
        self.buf=ReplayBuffer(); self.total_it=0

    def act(self, s, noise=0.1):
        st=torch.tensor(s.astype(np.float32), device=DEVICE).unsqueeze(0)
        acts=[]
        with torch.no_grad():
            for a in self.actors:
                u=a(st).cpu().numpy()[0,0]
                u=np.clip(u+np.random.normal(0,noise), -1, 1)
                acts.append(u)
        return np.array(acts)

    def _soft(self, net, net_t):
        for p,pt in zip(net.parameters(), net_t.parameters()):
            pt.data.copy_(self.tau*p.data+(1-self.tau)*pt.data)

    def update(self, batch=128):
        if len(self.buf)<batch: return
        self.total_it+=1
        tr=self.buf.sample(batch)
        S=torch.tensor(np.array([t[0] for t in tr],dtype=np.float32),device=DEVICE)
        A=torch.tensor(np.array([t[1] for t in tr],dtype=np.float32),device=DEVICE)
        R=torch.tensor(np.array([t[2] for t in tr],dtype=np.float32),device=DEVICE).unsqueeze(1)
        S2=torch.tensor(np.array([t[3] for t in tr],dtype=np.float32),device=DEVICE)
        D=torch.tensor(np.array([t[4] for t in tr],dtype=np.float32),device=DEVICE).unsqueeze(1)
        with torch.no_grad():
            A2=torch.cat([at(S2) for at in self.actors_t],dim=1)
            if self.algo=="matd3":
                noise=(torch.randn_like(A2)*self.target_noise).clamp(-self.noise_clip,self.noise_clip)
                A2=(A2+noise).clamp(-1,1)
                q1=self.critic1_t(S2,A2); q2=self.critic2_t(S2,A2)
                qt=torch.min(q1,q2)
            else:
                qt=self.critic1_t(S2,A2)
            y=R+self.gamma*(1-D)*qt
        q1=self.critic1(S,A); closs=F.mse_loss(q1,y)
        if self.algo=="matd3":
            q2=self.critic2(S,A); closs=closs+F.mse_loss(q2,y)
        self.c_opt.zero_grad(); closs.backward(); self.c_opt.step()
        # отложенное обновление актора (td3)
        if self.algo!="matd3" or self.total_it % self.policy_delay==0:
            for i in range(self.n):
                A_cur=A.clone()
                A_cur[:,i:i+1]=self.actors[i](S)
                aloss=-self.critic1(S,A_cur).mean()
                self.a_opts[i].zero_grad(); aloss.backward(); self.a_opts[i].step()
            for a,at in zip(self.actors,self.actors_t): self._soft(a,at)
            self._soft(self.critic1,self.critic1_t)
            if self.critic2 is not None: self._soft(self.critic2,self.critic2_t)

    def snapshot(self):
        return [copy.deepcopy(a.state_dict()) for a in self.actors]
    def load_snapshot(self, snap):
        for a,sd in zip(self.actors, snap): a.load_state_dict(sd)

print("OK: MARLTrainer (MADDPG/MATD3) загружен")

OK: MARLTrainer (MADDPG/MATD3) загружен


## 5. Цикл обучения: ранняя остановка + лучший чекпойнт
Мониторинг скользящего среднего валидационной награды; в финал загружается **лучший по награде** чекпойнт (Uncertainty-Guided Checkpoint Selection). Это ускоряет эксперименты — требование пользователя.

In [ ]:
# ---------------------------------------------------------------------------
# ЦИКЛ ОБУЧЕНИЯ: ранняя остановка + загрузка лучшего по награде чекпойнта
# ---------------------------------------------------------------------------
def evaluate_policy(trainer, env_fn, rollouts=ROLLOUTS, noise=0.0):
    """Средняя по роллаутам кумулятивная награда (детерминированная политика)."""
    rs=[]
    for _ in range(rollouts):
        env=env_fn(); s=env.reset(); done=False; total=0.0
        while not done:
            a=trainer.act(s, noise=noise); s,r,done,_=env.step(a); total+=r
        rs.append(total)
    return float(np.mean(rs)), float(np.std(rs))

def train_with_early_stopping(env_fn, algo="matd3", seed=0, max_episodes=MAX_EPISODES,
                              es_window=ES_WINDOW, es_patience=ES_PATIENCE,
                              val_fn=None, verbose=False, max_steps=None):
    """
    Обучает MADDPG/MATD3 с ранней остановкой по скользящему среднему ВАЛИДАЦИОННОЙ
    награды и сохраняет лучший по награде чекпойнт. В финал загружается ИМЕННО он
    (а не последняя политика) — это требование пользователя для ускорения.

    Метод ранней остановки опирается на Uncertainty-Guided Checkpoint Selection:
    мониторим скользящее среднее награды, лучший чекпойнт = max валидационной награды.
    """
    set_seed(seed)
    tr=MARLTrainer(env_fn, algo=algo, seed=seed)
    val_fn = val_fn or (lambda t: evaluate_policy(t, env_fn, rollouts=8)[0])
    rewards=[]; val_curve=[]
    best_val=-1e18; best_snap=tr.snapshot(); best_ep=0
    no_improve=0
    cap = max_steps or max_episodes
    for ep in range(cap):
        env=env_fn(); s=env.reset(); done=False; ep_r=0.0
        while not done:
            a=tr.act(s, noise=max(0.05, 0.3*(1-ep/cap)))   # затухающий шум исследования
            s2,r,done,_=env.step(a)
            tr.buf.add(s,a,r,s2,float(done)); s=s2; ep_r+=r
            tr.update()
        rewards.append(ep_r)
        # валидация раз в окно
        if (ep+1)%es_window==0:
            v=val_fn(tr); val_curve.append((ep+1, v))
            if v>best_val+1e-4:
                best_val=v; best_snap=tr.snapshot(); best_ep=ep+1; no_improve=0
            else:
                no_improve+=1
            if verbose: print(f"  [{algo} seed{seed}] ep{ep+1} val={v:.3f} best={best_val:.3f}")
            if no_improve>=es_patience:
                if verbose: print(f"  ⏹ ранняя остановка на ep{ep+1} (лучший ep{best_ep})")
                break
    tr.load_snapshot(best_snap)   # ЗАГРУЗКА ЛУЧШЕГО по награде
    return tr, dict(rewards=rewards, val_curve=val_curve, best_val=best_val,
                    best_ep=best_ep, stopped_ep=len(rewards))

print("OK: train_with_early_stopping + evaluate_policy загружены")

OK: train_with_early_stopping + evaluate_policy загружены


## 6. Самореферентный модуль (ядро ТЗ)
**Bilevel**: внешний уровень — эволюция геномов (reward-веса + интенсивность действий), внутренний — обучение MATD3. Кроссовер + мутации + адаптивная популяция 8→12. **Двойная страховка от reward hacking**: (1) разнообразие популяции; (2) hold-out фитнес + штраф за неправдоподобные траектории (СКР>2.5, скачки населения).

In [ ]:
# ---------------------------------------------------------------------------
# САМОРЕФЕРЕНТНЫЙ МОДУЛЬ для MATD3: эволюция reward-весов + интенсивности действий
# ---------------------------------------------------------------------------
@dataclass
class Genome:
    """Геном = конфигурация reward-функции + интенсивность действий (то, что
    самореферентный агент меняет В СЕБЕ). Это и есть «частичная самореферентность»."""
    weights: dict
    intensity: float
    fitness: float = -1e18
    val_fitness: float = -1e18           # на hold-out (anti-reward-hacking)
    plausibility_penalty: float = 0.0

    def vector(self):
        keys=sorted(self.weights.keys())
        return np.array([self.weights[k] for k in keys]+[self.intensity]), keys
    @staticmethod
    def from_vector(vec, keys):
        w={k:float(vec[i]) for i,k in enumerate(keys)}
        return Genome(weights=w, intensity=float(vec[len(keys)]))

def random_genome(rng):
    w=dict(REWARD_WEIGHTS_INIT)
    for k in w:
        w[k]=float(w[k]*(1+rng.normal(0,0.3)))           # вариация вокруг курских весов
    w["action_cost"]=-abs(w["action_cost"])               # штраф остаётся отрицательным
    return Genome(weights=w, intensity=float(np.clip(rng.normal(1.0,0.25),0.3,2.0)))

def crossover(g1, g2, rng):
    v1,keys=g1.vector(); v2,_=g2.vector()
    mask=rng.random(len(v1))<0.5
    child=np.where(mask, v1, v2)
    return Genome.from_vector(child, keys)

def mutate(g, rng, rate=0.3, scale=0.15):
    v,keys=g.vector()
    for i in range(len(v)):
        if rng.random()<rate: v[i]*=(1+rng.normal(0,scale))
    g2=Genome.from_vector(v, keys)
    g2.weights["action_cost"]=-abs(g2.weights["action_cost"])
    g2.intensity=float(np.clip(g2.intensity,0.3,2.0))
    return g2

def plausibility_penalty(panel_df, trainer, env_fn, col_index):
    """
    ANTI-REWARD-HACKING (механизм 2): штраф за демографически НЕВОЗМОЖНЫЕ траектории.
    Прогоняем обученную политику и штрафуем за: СКР>2.5, население<=0, скачки сальдо,
    нефизичные изменения год-к-году. Если агент «взломал» world-model — здесь всплывёт.
    """
    env=env_fn(); s=env.reset(); done=False
    pen=0.0; prev=s.copy()
    tfr_j=col_index.get("СКР (всего)"); pop_j=col_index.get("Численность населения всего")
    while not done:
        a=trainer.act(s, noise=0.0); s,r,done,_=env.step(a)
        if tfr_j is not None and s[tfr_j]>TFR_CAP+1e-3: pen+=s[tfr_j]-TFR_CAP
        if pop_j is not None and s[pop_j]<=0: pen+=5.0
        # скачок населения год-к-году > 8%
        if pop_j is not None and prev[pop_j]>0:
            jump=abs(s[pop_j]-prev[pop_j])/prev[pop_j]
            if jump>0.08: pen+=(jump-0.08)*10
        prev=s.copy()
    return float(pen)

def evolve_self_referential(panel_df, world_model, col_index, seed=0,
                            pop_min=POP_MIN, pop_max=POP_MAX, generations=6,
                            inner_steps=120, holdout_split=0.3, verbose=False,
                            show_progress=True):
    """
    Самореферентный MATD3 по ОДНОМУ региону (отдельный эксперимент на регион).
    Внешний уровень (эволюция): популяция геномов reward+intensity, селекция/кроссовер/мутации.
    Внутренний уровень: MATD3 учится при данном геноме с ранней остановкой.
    """
    rng=np.random.default_rng(seed); set_seed(seed)
    yrs=sorted(panel_df[panel_df["Год"]>=2025]["Год"].unique())
    cut=int(len(yrs)*(1-holdout_split))
    train_h=(yrs[0], yrs[cut-1]); hold_h=(yrs[cut], yrs[-1])

    def make_env(genome, horizon):
        return lambda: DemographyEnv(panel_df, world_model, col_index,
                                     reward_weights=genome.weights,
                                     action_intensity=genome.intensity,
                                     horizon_years=horizon)

    pop=[random_genome(rng) for _ in range(pop_min)]
    history=[]; best_overall=None; stale=0

    gen_iter = pbar(range(generations), total=generations,
                    desc="Self-ref generations",
                    leave=False, disable=not show_progress)

    for gen in gen_iter:
        eval_candidates = [g for g in pop if g.fitness <= -1e17]
        cand_iter = pbar(eval_candidates, total=len(eval_candidates),
                         desc=f"Gen {gen+1}/{generations}",
                         leave=False, disable=not show_progress)

        for g in cand_iter:
            env_tr=make_env(g, train_h)
            tr,log=train_with_early_stopping(env_tr, algo="matd3", seed=seed+gen,
                                             max_steps=inner_steps, es_window=15, es_patience=3)
            env_ho=make_env(g, hold_h)
            val_mean,_=evaluate_policy(tr, env_ho, rollouts=10)
            pen=plausibility_penalty(panel_df, tr, env_ho, col_index)
            g.val_fitness=val_mean; g.plausibility_penalty=pen
            g.fitness=val_mean - 0.5*pen
            g._trainer=tr

            if show_progress:
                cand_iter.set_postfix_str(
                    f"fit={g.fitness:.3f} | val={g.val_fitness:.3f} | pen={g.plausibility_penalty:.3f}"
                )

        pop.sort(key=lambda x:x.fitness, reverse=True)
        gen_best=pop[0]

        if best_overall is None or gen_best.fitness>best_overall.fitness:
            best_overall=gen_best; stale=0
        else:
            stale+=1

        history.append(dict(gen=gen, best_fitness=gen_best.fitness,
                            best_val=gen_best.val_fitness, penalty=gen_best.plausibility_penalty,
                            pop_size=len(pop), weights=dict(gen_best.weights),
                            intensity=gen_best.intensity))

        if show_progress:
            gen_iter.set_postfix_str(
                f"best={gen_best.fitness:.3f} | val={gen_best.val_fitness:.3f} | "
                f"pen={gen_best.plausibility_penalty:.3f} | pop={len(pop)} | stale={stale}"
            )
        elif verbose:
            print(f" поколение {gen}: best_fit={gen_best.fitness:.3f} "
                  f"val={gen_best.val_fitness:.3f} pen={gen_best.plausibility_penalty:.3f} pop={len(pop)}")

        if stale>=2 and len(pop)<pop_max:
            need=min(2, pop_max-len(pop))
            pop.extend([random_genome(rng) for _ in range(need)])
            stale=0

        survivors=pop[:max(2, len(pop)//2)]
        children=[]
        while len(survivors)+len(children)<len(pop):
            p1,p2=rng.choice(survivors, 2, replace=True)
            ch=mutate(crossover(p1,p2,rng), rng)
            children.append(ch)
        pop=survivors+children
        for g in pop[len(survivors):]:
            g.fitness=-1e18

    return best_overall, history

print("OK: самореферентный модуль (evolve_self_referential) загружен")

OK: самореферентный модуль (evolve_self_referential) загружен


## 7. Мульти-сид + 95% ДИ + загрузка панелей
Критерий **доказанного эффекта** (утверждён пользователем): бутстрэп 95% ДИ изменения метрики 2050 **не пересекает ноль**.

In [ ]:
# ---------------------------------------------------------------------------
# МУЛЬТИ-СИД + 95% ДИ (критерий доказанного эффекта) + загрузка панелей
# ---------------------------------------------------------------------------
import pandas as pd

def load_panel(path):
    """Загружает CCM-панель (выход Ноутбука 1) и строит col_index имя->позиция."""
    df=pd.read_csv(path)
    cols=[c for c in df.columns if c not in ("Год","Регион","Тип половозрастной структуры (Сундберг)")]
    col_index={c:i for i,c in enumerate(cols)}
    return df, col_index

def build_world_model(panel_df, col_index, seed=0):
    """Обучает world-model на исторических переходах CCM-панели (1991-2024)."""
    set_seed(seed)
    cols=list(col_index.keys())
    hist=panel_df[panel_df["Год"]<=2024].sort_values("Год")
    M=hist[cols].to_numpy().astype(float)
    if len(M)<3:
        M=panel_df[cols].to_numpy().astype(float)
    S=M[:-1]; S2=M[1:]
    A=np.zeros((len(S), N_AGENTS))   # исторически действий нет -> нулевое воздействие
    wm=WorldModelEnsemble(state_dim=len(cols), action_dim=N_AGENTS)
    wm.fit(S, A, S2, epochs=30)
    return wm

def bootstrap_ci(values, n_boot=5000, alpha=0.05, seed=0):
    """Бутстрэп 95% ДИ среднего. Эффект ДОКАЗАН, если ДИ не пересекает 0."""
    rng=np.random.default_rng(seed); arr=np.array(values, dtype=float)
    means=[rng.choice(arr, len(arr), replace=True).mean() for _ in range(n_boot)]
    lo,hi=np.percentile(means,[100*alpha/2, 100*(1-alpha/2)])
    return float(arr.mean()), float(lo), float(hi)

def metric_change_vs_baseline(trainer, env_fn, col_index, metric="Численность населения всего"):
    """Изменение целевой метрики 2050 относительно CCM-baseline, % (для статистики)."""
    env=env_fn(); s=env.reset(); done=False
    while not done:
        a=trainer.act(s, noise=0.0); s,r,done,_=env.step(a)
    j=col_index[metric]; base=env.baseline[env.y1][j]
    return (s[j]-base)/(abs(base)+1e-6)*100.0

def run_multiseed(panel_df, col_index, algo="matd3", n_seeds=N_SEEDS,
                  max_steps=None, metric="Численность населения всего", verbose=False,
                  show_progress=True):
    """
    Мульти-сид прогон одной конфигурации. Возвращает изменения метрики по сидам +
    95% ДИ. Если ДИ не пересекает 0 — эффект ДОКАЗАН (критерий пользователя).
    """
    changes=[]; rewards=[]; stops=[]
    seed_iter = pbar(range(n_seeds), total=n_seeds,
                     desc=f"{algo.upper()} seeds",
                     leave=False, disable=not show_progress)
    for sd in seed_iter:
        wm=build_world_model(panel_df, col_index, seed=sd)
        env_fn=lambda: DemographyEnv(panel_df, wm, col_index)
        tr,log=train_with_early_stopping(env_fn, algo=algo, seed=sd, max_steps=max_steps)
        ch=metric_change_vs_baseline(tr, env_fn, col_index, metric=metric)

        changes.append(ch); rewards.append(log["best_val"]); stops.append(log["stopped_ep"])

        if show_progress:
            seed_iter.set_postfix_str(
                f"Δ={ch:+.2f}% | best={log['best_val']:.3f} | stop={log['stopped_ep']}"
            )
        elif verbose:
            print(f" [{algo}] seed {sd}: Δ{metric}={ch:+.2f}% (стоп ep{log['stopped_ep']})")

    mean,lo,hi=bootstrap_ci(changes)
    proven = (lo>0) or (hi<0) # 95% ДИ не пересекает 0
    return dict(algo=algo, metric=metric, changes=changes, mean=mean, ci_low=lo, ci_high=hi,
                proven_effect=bool(proven), avg_stop=float(np.mean(stops)),
                mean_best_reward=float(np.mean(rewards)))

print("OK: мульти-сид + bootstrap_ci + загрузка панелей загружены")

OK: мульти-сид + bootstrap_ci + загрузка панелей загружены


## 8. Финал: общий частично самореферентный агент
Стартует с усреднённого по регионам лучшего эволюционного генома (перенос знаний), дообучается на смеси сред всех регионов, оценивается по каждому региону.

In [ ]:
# ---------------------------------------------------------------------------
# ФИНАЛ: общий частично самореферентный агент, управляющий ВСЕМИ регионами
# ---------------------------------------------------------------------------
def build_general_agent(panels, evolved_configs, seed=0, inner_steps=200, verbose=False,
                        show_progress=True):
    """
    Общий агент обучается на ОПЫТЕ прошлых экспериментов:
    • стартует с УСРЕДНЁННОГО по регионам лучшего эволюционного генома
    (reward-веса + интенсивность) — перенос знаний;
    • дообучается на смеси сред всех регионов (общая политика);
    • остаётся частично самореферентным: может донастроить геном на общем опыте.
    """
    set_seed(seed)
    keys=sorted(REWARD_WEIGHTS_INIT.keys())
    W=np.mean([[g.weights[k] for k in keys] for g in evolved_configs.values()], axis=0)
    I=float(np.mean([g.intensity for g in evolved_configs.values()]))
    shared_weights={k:float(W[i]) for i,k in enumerate(keys)}

    region_list=list(panels.keys())

    wm_iter = pbar(region_list, total=len(region_list),
                   desc="General agent | world models",
                   leave=False, disable=not show_progress)
    wms={}
    for r in wm_iter:
        wms[r] = build_world_model(panels[r][0], panels[r][1], seed=seed)
        if show_progress:
            wm_iter.set_postfix_str(f"ready={r}")

    def mixed_env_fn():
        r=region_list[np.random.randint(len(region_list))]
        df,ci=panels[r]
        return DemographyEnv(df, wms[r], ci, reward_weights=shared_weights, action_intensity=I)

    tr,log=train_with_early_stopping(mixed_env_fn, algo="matd3", seed=seed, max_steps=inner_steps)

    per_region={}
    eval_iter = pbar(region_list, total=len(region_list),
                     desc="General agent | final eval",
                     leave=False, disable=not show_progress)
    for r in eval_iter:
        df,ci=panels[r]
        env_fn=lambda df=df, ci=ci, r=r: DemographyEnv(df, wms[r], ci,
                                                       reward_weights=shared_weights, action_intensity=I)
        ch=metric_change_vs_baseline(tr, env_fn, ci, metric="Численность населения всего")
        per_region[r]=ch
        if show_progress:
            eval_iter.set_postfix_str(f"{r}: {ch:+.2f}%")
        elif verbose:
            print(f" {r}: Δнаселение 2050 = {ch:+.2f}%")

    return tr, dict(shared_weights=shared_weights, intensity=I,
                    per_region_change=per_region, train_log=log)

print("OK: build_general_agent (общий агент) загружен")

OK: build_general_agent (общий агент) загружен


## 9. Полный прогон эксперимента
Запускает все этапы по 9 регионам. **Внимание:** тяжёлый прогон — рассчитан на Colab T4.
Этапы сохраняют промежуточные результаты в `exp_results/` для resume между сессиями.

In [ ]:
import os, json, pandas as pd, numpy as np
PANELS_DIR = "/content/"           # выход Ноутбука 1
OUT = "exp_results"; os.makedirs(OUT, exist_ok=True)

REGIONS = ["Республика_Алтай","Республика_Бурятия","Республика_Тыва","Республика_Хакасия",
           "Алтайский_край","Забайкальский_край","Красноярский_край",
           "Иркутская_область","Новосибирская_область"]

# загрузка всех панелей
panels = {}
for r in REGIONS:
    p = os.path.join(PANELS_DIR, f"panel_{r}.csv")
    if os.path.exists(p):
        df, ci = load_panel(p); panels[r] = (df, ci)
    else:
        print(f"⚠ нет панели: {p}")
print(f"Загружено регионов: {len(panels)}")

Загружено регионов: 9


### 9.1. Базовые прогоны MADDPG и MATD3 (мульти-сид, 10 сидов)
По каждому региону считаем изменение населения 2050 и 95% ДИ.

In [ ]:
baseline_results = {}
region_iter = pbar(list(panels.items()), total=len(panels),
                   desc="9.1 Baselines by region", leave=True)

for r,(df,ci) in region_iter:
    region_iter.set_postfix_str(r)

    res_maddpg = run_multiseed(df, ci, algo="maddpg", n_seeds=N_SEEDS,
                               verbose=False, show_progress=True)
    res_matd3 = run_multiseed(df, ci, algo="matd3", n_seeds=N_SEEDS,
                              verbose=False, show_progress=True)

    baseline_results[r] = dict(maddpg=res_maddpg, matd3=res_matd3)

    json.dump(baseline_results, open(os.path.join(OUT,"baseline_results.json"),"w"),
              ensure_ascii=False, indent=2, default=float)

print("\n✓ Базовые прогоны сохранены: exp_results/baseline_results.json")

9.1 Baselines by region:   0.00%|          | 0/9 [00:00<?, ?it/s]

MADDPG seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]

MATD3 seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]

MADDPG seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]

MATD3 seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]

...

MADDPG seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]

MATD3 seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]

MADDPG seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]

MATD3 seeds:   0.00%|          | 0/10 [00:00<?, ?it/s]


✓ Базовые прогоны сохранены: exp_results/baseline_results.json


### 9.2. Самореферентный MATD3 по каждому региону (эволюция reward+интенсивность)
Отдельный эволюционный прогон на регион. Тяжёлый этап — сохраняйте результаты для resume.

In [ ]:
evolved_configs = {}; evolution_history = {}
region_iter = pbar(list(panels.items()), total=len(panels),
                   desc="9.2 Self-ref MATD3 by region", leave=True)

for r,(df,ci) in region_iter:
    region_iter.set_postfix_str(r)

    wm = build_world_model(df, ci, seed=0)
    best, hist = evolve_self_referential(df, wm, ci, seed=0,
                                         pop_min=POP_MIN, pop_max=POP_MAX,
                                         generations=6, inner_steps=120,
                                         verbose=False, show_progress=True)
    evolved_configs[r] = best
    evolution_history[r] = hist

    json.dump({k:dict(weights=v.weights, intensity=v.intensity, fitness=v.fitness,
                      val_fitness=v.val_fitness, penalty=v.plausibility_penalty)
               for k,v in evolved_configs.items()},
              open(os.path.join(OUT,"evolved_configs.json"),"w"),
              ensure_ascii=False, indent=2, default=float)

    json.dump(evolution_history, open(os.path.join(OUT,"evolution_history.json"),"w"),
              ensure_ascii=False, indent=2, default=float)

print("\n✓ Эволюционные конфиги сохранены: exp_results/evolved_configs.json")

9.2 Self-ref MATD3 by region:   0.00%|          | 0/9 [00:00<?, ?it/s]

Self-ref generations:   0.00%|          | 0/6 [00:00<?, ?it/s]

Gen 1/6:   0.00%|          | 0/8 [00:00<?, ?it/s]

Gen 2/6:   0.00%|          | 0/4 [00:00<?, ?it/s]

Gen 3/6:   0.00%|          | 0/4 [00:00<?, ?it/s]

...

Gen 3/6:   0.00%|          | 0/4 [00:00<?, ?it/s]

Gen 4/6:   0.00%|          | 0/5 [00:00<?, ?it/s]

Gen 5/6:   0.00%|          | 0/5 [00:00<?, ?it/s]

Gen 6/6:   0.00%|          | 0/6 [00:00<?, ?it/s]


✓ Эволюционные конфиги сохранены: exp_results/evolved_configs.json


### 9.3. Финал — общий агент на опыте всех регионов

In [ ]:
tr_general, general_res = build_general_agent(panels, evolved_configs, seed=0,
                                              inner_steps=200, verbose=False,
                                              show_progress=True)

json.dump(dict(shared_weights=general_res["shared_weights"], intensity=general_res["intensity"],
               per_region_change=general_res["per_region_change"]),
          open(os.path.join(OUT,"general_agent.json"),"w"),
          ensure_ascii=False, indent=2, default=float)

print("✓ Общий агент сохранён: exp_results/general_agent.json")

General agent | world models:   0.00%|          | 0/9 [00:00<?, ?it/s]

General agent | final eval:   0.00%|          | 0/9 [00:00<?, ?it/s]

✓ Общий агент сохранён: exp_results/general_agent.json


### 9.4. Сводное сравнение всех результатов + проверка критерия эффекта
Таблица: регион × метод × Δнаселение 2050 × 95% ДИ × «эффект доказан».

In [ ]:
rows = []
for r in panels:
    for algo in ["maddpg","matd3"]:
        res = baseline_results[r][algo]
        rows.append(dict(Регион=r, Метод=algo.upper(),
                         Δ_население_2050=round(res["mean"],2),
                         ДИ95_низ=round(res["ci_low"],2), ДИ95_верх=round(res["ci_high"],2),
                         Эффект_доказан=res["proven_effect"], Средний_стоп=round(res["avg_stop"],0)))
    if r in evolved_configs:
        g = evolved_configs[r]
        rows.append(dict(Регион=r, Метод="MATD3+самореф",
                         Δ_население_2050="—", ДИ95_низ="—", ДИ95_верх="—",
                         Эффект_доказан=f"fitness={g.fitness:.2f}", Средний_стоп="—"))
for r,ch in general_res["per_region_change"].items():
    rows.append(dict(Регион=r, Метод="Общий агент", Δ_население_2050=round(ch,2),
                     ДИ95_низ="—", ДИ95_верх="—", Эффект_доказан="—", Средний_стоп="—"))
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT,"summary_comparison.csv"), index=False, encoding="utf-8-sig")
print("✓ Сводка: exp_results/summary_comparison.csv")
summary

NameError: name 'baseline_results' is not defined

In [ ]:
import os
import shutil
from datetime import datetime
from google.colab import files, drive

# =========================================
# 1) Архивация /content/exp_results
# =========================================
SRC_DIR = "/content/exp_results"

if not os.path.exists(SRC_DIR):
    raise FileNotFoundError(f"Папка не найдена: {SRC_DIR}")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = f"/content/exp_results_{stamp}"
zip_path = shutil.make_archive(zip_base, "zip", SRC_DIR)

print(f"✓ Архив создан: {zip_path}")
print(f"✓ Размер архива: {round(os.path.getsize(zip_path) / 1024 / 1024, 2)} MB")

# =========================================
# 2) Этап 1: автоскачивание на локальный ПК
# =========================================
print("▶ Запускаю скачивание архива на локальный компьютер...")
files.download(zip_path)

# =========================================
# 3) Этап 2: сохранение этого же архива на Google Drive
# =========================================
print("▶ Монтирую Google Drive...")
drive.mount('/content/drive')

drive_dir = "/content/drive/MyDrive/colab_archives"
os.makedirs(drive_dir, exist_ok=True)

drive_zip_path = os.path.join(drive_dir, os.path.basename(zip_path))
shutil.copy2(zip_path, drive_zip_path)

print(f"✓ Архив сохранён на Google Drive: {drive_zip_path}")
print("✓ Готово")